# PARISO: Pixelated RIS Unit-Cell Phase–Frequency Response Dataset Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the PARISO dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.spe2-z45q/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their @id
record_sets = metadata.recordSet if hasattr(metadata, 'recordSet') else []
if not record_sets:
    print("No record sets found at the top level. Attempting to enumerate inner structure...")

# If dataset.recordSet is empty, explore dataset.distribution to find embedded record sets
record_set_ids = []
distribution = getattr(metadata, 'distribution', [])
for dist in distribution:
    # Each distribution may contain its own metadata
    dist_id = dist['@id'] if isinstance(dist, dict) and '@id' in dist else dist
    try:
        # Open as another Dataset object to inspect nested recordSets
        dist_dataset = mlc.Dataset(dist_id)
        dist_metadata = dist_dataset.metadata
        if hasattr(dist_metadata, 'recordSet') and dist_metadata.recordSet:
            for rs in dist_metadata.recordSet:
                # Print out RecordSet and its fields
                print(f"RecordSet @id: {rs['@id'] if isinstance(rs, dict) else rs}")
                if isinstance(rs, dict) and 'field' in rs:
                    fields = rs['field']
                    if isinstance(fields, list):
                        for f in fields:
                            print(f"  Field @id: {f['@id'] if isinstance(f, dict) else f}")
                    else:
                        print(f"  Field @id: {fields}")
                record_set_ids.append(rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs)
        else:
            print(f"Distribution {dist_id} has no recordSet found.")
    except Exception as e:
        print(f"Warning: Could not inspect distribution {dist_id}: {e}")
        continue

# If still no recordSets found, try some common names as in Croissant conventions
if not record_set_ids:
    print("No record sets discovered inside distribution metadata. Try typical names: 'axis_definitions', 'simulation_outputs', 'upload_manifest'")
    record_set_ids = ['axis_definitions', 'simulation_outputs', 'upload_manifest']  # Example fallback for demonstration

# Print discovered record_set_ids
print("Discovered record sets IDs:")
for rsid in record_set_ids:
    print(rsid)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# Use discovered record_set_ids from previous cell
dataframes = {}
for record_set_id in record_set_ids:
    try:
        # Attempt to load records using mlcroissant
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded record set '{record_set_id}' with columns:")
            print(df.columns.tolist())
            print(df.head())
        else:
            print(f"No records found for record set '{record_set_id}'.")
    except Exception as e:
        print(f"Error loading record set '{record_set_id}': {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section includes operations such as removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a record set for analysis. Replace with record_set_id for your dataset.
analyze_record_set_id = None
for key in dataframes.keys():
    if not analyze_record_set_id:
        analyze_record_set_id = key
        break

if analyze_record_set_id:
    df = dataframes[analyze_record_set_id].copy()
    print(f"Analyzing record set: {analyze_record_set_id}")
    # Identify numeric fields
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields: {numeric_columns}")
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        # Example threshold filter
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt grouping by another field
        possible_group_fields = [col for col in df.columns if col != numeric_field_id]
        group_field_id = possible_group_fields[0] if possible_group_fields else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found in record set for EDA.")
else:
    print("No suitable record set found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization: Histogram and scatter plot for numeric columns
if analyze_record_set_id and numeric_columns:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if len(numeric_columns) > 1:
        plt.figure(figsize=(6, 6))
        plt.scatter(df[numeric_columns[0]], df[numeric_columns[1]], alpha=0.5)
        plt.title(f"Scatter plot: {numeric_columns[0]} vs {numeric_columns[1]}")
        plt.xlabel(numeric_columns[0])
        plt.ylabel(numeric_columns[1])
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The PARISO dataset was successfully loaded and record sets were explored using their `@id`.
- Numeric fields were filtered and normalized, and grouped by key attributes for basic analysis.
- Visualizations revealed data distribution and relationships within numeric fields.

Further analysis could include spatial/temporal exploration, or use ML models for predictive benchmarking on the extracted features.